# Template: Pipeline-Ready Notebook

This notebook serves as a **template** for creating analysis steps that integrate seamlessly with the project's DVC pipeline.

**Key Requirements for Pipeline Notebooks:**
1.  **No Hardcoded Paths**: Always use `ProjectPaths` from `scripts.paths`.
2.  **Parameters Cell**: Dedicate a cell for parameters (tagged `parameters`) so `dvc.yaml` can inject values.
3.  **Deterministic Outputs**: Always save results to `data/processed` or `figures/` using standard names.

In [ ]:
# --- 1. Imports ---
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Ensure we can import from 'scripts' directory
# This allows running the notebook from anywhere (VS Code, terminal, etc.)
project_root = Path("../../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils.paths import ProjectPaths

## 2. Parameters (The "Control Panel")

**CRITICAL:** This cell receives values from `dvc.yaml` / `params.yaml`.
To enable this:
1.  Click this cell.
2.  In the right sidebar (Property Inspector), add the tag **`parameters`**.
3.  Define default values here. They will be overwritten during `dvc repro` if specified.

In [ ]:
# --- Parameters ---
# Default values (used when running interactively)
plot_color = 'blue'
filter_threshold_std = 2.0
example_param = 42

## 3. Data Loading

Use `ProjectPaths` to load inputs. This ensures that if the raw data location changes, you only update it in one place (`scripts/paths.py`).

In [ ]:
# Example: Loading the PCA Scores from the previous step
input_path = ProjectPaths.PCA_SCORES_FILE

if input_path.exists():
    df = pd.read_csv(input_path, index_col=0)
    print(f"Loaded data from {input_path}")
    display(df.head())
else:
    print(f"Warning: Input file {input_path} not found. (Run previous pipeline steps first)")
    # Create dummy data for template demonstration
    df = pd.DataFrame({'PC1': [1, 2, 3], 'PC2': [4, 5, 6]})

## 4. Analysis Logic

Perform your transformations, modeling, or plotting here. Use the parameters defined above.

In [ ]:
# Apply logic using the parameter
print(f"Filtering data with threshold: {filter_threshold_std} (std dev)")
print(f"Using color: {plot_color}")

plt.figure(figsize=(6, 4))
plt.scatter(df['PC1'], df['PC2'], c=plot_color, alpha=0.6)
plt.title(f"Example Plot (Param: {example_param})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
# plt.show() # Optional in pipeline, but good for interactive check

## 5. Saving Outputs

Save processed data or figures to the directories managed by `ProjectPaths`.
**Do not** save to random locations or use hardcoded paths like `C:/Users/...`.

In [ ]:
# Ensure directories exist (good practice)
ProjectPaths.ensure_directories()

# 1. Save Figure
fig_out = ProjectPaths.FIGURES_DIR / "example_pipeline_plot.png"
plt.savefig(fig_out)
print(f"Saved figure to: {fig_out}")

# 2. Save Data
# Ideally, define this path in ProjectPaths if it's a critical pipeline artifact
data_out = ProjectPaths.PROCESSED_DATA_DIR / "example_pipeline_output.csv"
df.to_csv(data_out)
print(f"Saved data to: {data_out}")